In [72]:
import pandas as pd
import plotly.express as px

### Load the selected dataset

The CSV was produced by the cleaning pipeline and contains the selected BRFSS predictors plus the heart disease/attack outcome. The first rows provide a quick check that the file loaded and the columns look as expected. Each row represents one respondent, not a longitudinal record.

In [73]:
DATA_PATH = '../../data'
brfss_df_selected = pd.read_csv(f'{DATA_PATH}/processed/heart_disease_health_indicators_BRFSS2015_with_meanings.csv')
brfss_df_selected.head()

,HeartDiseaseorAttack,InterviewDate,InterviewMonth,InterviewDay,InterviewYear,HighBP,HighChol,CholCheck,BMI,Smoker,...,HvyAlcoholConsump_Meaning,AnyHealthcare_Meaning,NoDocbcCost_Meaning,GenHlth_Meaning,DiffWalk_Meaning,Sex_Meaning,Age_Meaning,Education_Meaning,Income_Meaning,State_Meaning
0,0.0,2015-01-29,1,29,2015,1.0,1.0,1.0,40.18,1.0,...,No heavy drinking,Has health-care coverage,Did not need to see a doctor but could not aff...,Poor,Serious difficulty walking or climbing stairs,Female,60–64,Grade 12 or GED,"$15,000 to <$20,000",Alabama
1,0.0,2015-01-20,1,20,2015,0.0,0.0,0.0,25.09,1.0,...,No heavy drinking,No health-care coverage,Could not afford to see a doctor in past 12 mo...,Good,No serious difficulty walking or climbing stairs,Female,50–54,College graduate: 4 or more years,"Less than $10,000",Alabama
2,0.0,2015-01-14,1,14,2015,1.0,1.0,1.0,28.19,0.0,...,No heavy drinking,Has health-care coverage,Could not afford to see a doctor in past 12 mo...,Poor,Serious difficulty walking or climbing stairs,Female,60–64,Grade 12 or GED,"$75,000 or more",Alabama
3,0.0,2015-01-14,1,14,2015,1.0,0.0,1.0,26.52,0.0,...,No heavy drinking,Has health-care coverage,Did not need to see a doctor but could not aff...,Very good,No serious difficulty walking or climbing stairs,Female,70–74,Grades 9–11,"$35,000 to <$50,000",Alabama
4,0.0,2015-01-05,1,5,2015,1.0,1.0,1.0,23.89,0.0,...,No heavy drinking,Has health-care coverage,Did not need to see a doctor but could not aff...,Very good,No serious difficulty walking or climbing stairs,Female,70–74,College/technical school: 1–3 years,"$20,000 to <$25,000",Alabama


In [74]:
brfss_df_selected.shape

(253680, 48)

# Exploratory data analysis: BRFSS heart health

This notebook explores the selected 2015 BRFSS dataset. `HeartDiseaseorAttack` is the binary outcome. The fields mix binary indicators, ordered survey categories, and numeric measures; interpret numeric summaries in light of their coding. These unweighted descriptive patterns do not establish causation.

The next cell derives a compact structural view. `dtype` is the pandas storage type, `missing_n` and `missing_pct` quantify unavailable entries, and `unique_n` counts distinct observed values. `describe()` gives numeric summaries; for survey codes these are arithmetic summaries of the codes and may not have a direct real-world unit.

## Dataset structure and data quality

In [75]:
profile = pd.DataFrame({
    'dtype': brfss_df_selected.dtypes.astype(str),
    'missing_n': brfss_df_selected.isna().sum(),
    'missing_pct': (brfss_df_selected.isna().mean() * 100).round(3),
    'unique_n': brfss_df_selected.nunique(dropna=True)
}).sort_values('missing_pct', ascending=False)
display(profile)
display(brfss_df_selected.describe().T)

,dtype,missing_n,missing_pct,unique_n
HeartDiseaseorAttack,float64,0,0.0,2
InterviewDate,str,0,0.0,411
State,float64,0,0.0,53
BMICategory,str,0,0.0,4
HeartDiseaseorAttack_Meaning,str,0,0.0,2
HighBP_Meaning,str,0,0.0,2
HighChol_Meaning,str,0,0.0,2
CholCheck_Meaning,str,0,0.0,2
Smoker_Meaning,str,0,0.0,2
Stroke_Meaning,str,0,0.0,2


,count,mean,std,min,25%,50%,75%,max
HeartDiseaseorAttack,253680.0,0.094186,0.292087,0.00,0.00,0.00,0.00,1.00
InterviewMonth,253680.0,6.423179,3.491804,1.00,3.00,6.00,10.00,12.00
InterviewDay,253680.0,14.205365,8.229681,1.00,7.00,14.00,21.00,31.00
InterviewYear,253680.0,2015.022205,0.147350,2015.00,2015.00,2015.00,2015.00,2016.00
HighBP,253680.0,0.429001,0.494934,0.00,0.00,0.00,1.00,1.00
HighChol,253680.0,0.424121,0.494210,0.00,0.00,0.00,1.00,1.00
CholCheck,253680.0,0.962670,0.189571,0.00,1.00,1.00,1.00,1.00
BMI,253680.0,28.398183,6.601811,12.02,24.14,27.34,31.31,97.65
Smoker,253680.0,0.443169,0.496761,0.00,0.00,0.00,1.00,1.00
Stroke,253680.0,0.040571,0.197294,0.00,0.00,0.00,0.00,1.00


The profile table distinguishes sparse missingness from variables where missing values may affect many records. The chart displays columns with at least one missing value, sorted by missing percentage. Missingness can reflect unanswered or unavailable survey responses; it should not be interpreted as a negative answer.

In [76]:
missing = profile.query('missing_n > 0').sort_values('missing_pct')
px.bar(missing, x='missing_pct', y=missing.index, orientation='h', title='Missingness by variable', labels={'missing_pct':'Missing values (%)', 'index':'Variable'}).show()

## Outcome prevalence and feature distributions

This section measures the outcome distribution. In `HeartDiseaseorAttack`, 0 means the respondent did not report heart disease or a heart attack and 1 means they did. The table shows counts and percentages; the bar chart makes class imbalance visible.

In [77]:
target = 'HeartDiseaseorAttack'

In [78]:
outcome_counts = brfss_df_selected[target].value_counts(dropna=False).sort_index()
outcome_summary = pd.DataFrame({'count': outcome_counts, 'percent': outcome_counts / len(brfss_df_selected) * 100})
display(outcome_summary.round(2))
px.bar(outcome_summary.reset_index(names='outcome'), x='outcome', y='count', text='percent', title='Heart disease or attack outcome balance', labels={'outcome':'Outcome (0 = no, 1 = yes)', 'count':'Respondents', 'percent':'Percent'}).show()

,count,percent
HeartDiseaseorAttack,,
0.0,229787,90.58
1.0,23893,9.42


In [79]:
other_cols = ['InterviewDate', 'InterviewDay', 'InterviewMonth', 'InterviewYear', 'MentHlth', 'PhysHlth', 'BMICategory', 'State', 'Income', 'BMI', 'Age']
cate_cols = [col for col in brfss_df_selected.columns if col not in other_cols + [target] and not col.endswith('_Meaning')]
cate_cols

['HighBP',
 'HighChol',
 'CholCheck',
 'Smoker',
 'Stroke',
 'Diabetes',
 'PhysActivity',
 'Fruits',
 'Veggies',
 'HvyAlcoholConsump',
 'AnyHealthcare',
 'NoDocbcCost',
 'GenHlth',
 'DiffWalk',
 'Sex',
 'Education']

In [80]:
binary_features = [c for c in cate_cols if c != target]
binary_prevalence = brfss_df_selected[binary_features].mean().mul(100).sort_values().rename('percent_yes')
binary_prevalence

Stroke                 4.057080
HvyAlcoholConsump      5.619678
NoDocbcCost            8.417692
DiffWalk              16.822375
Diabetes              29.692132
HighChol              42.412094
HighBP                42.900110
Smoker                44.316856
Fruits                63.425576
PhysActivity          75.654368
Veggies               81.141990
AnyHealthcare         95.105251
CholCheck             96.266950
Sex                  155.965784
GenHlth              251.139231
Education            505.043362
Name: percent_yes, dtype: float64

For binary indicators coded 0/1, the column mean equals the share coded 1. This chart shows how common each recorded indicator is in the full dataset. It does not show the outcome rate for that group; that comparison follows below.

In [81]:
px.bar(binary_prevalence, x='percent_yes', orientation='h', title='Prevalence of binary indicators', labels={'percent_yes':'Respondents coded 1 (%)'}).show()

## How recorded characteristics vary with the outcome

Outcome rates below are unadjusted descriptive associations. Group counts are included to show the size of each comparison.

For each binary predictor, this table and chart compare the fraction with `HeartDiseaseorAttack = 1` among records coded 0 versus 1. `n` is the number of records in that group. Keep group sizes in view: rates from small groups are less stable. These unadjusted differences may reflect age or other characteristics rather than an independent relationship.

In [82]:
association_rows = []
for col in binary_features:
    grouped = brfss_df_selected.groupby(col, dropna=False)[target].agg(['mean', 'count'])
    for category, row in grouped.iterrows():
        association_rows.append({'feature': col, 'category': category, 'outcome_rate_pct': row['mean'] * 100, 'n': row['count']})
binary_outcome_rates = pd.DataFrame(association_rows)
display(binary_outcome_rates.sort_values(['feature','category']).round(2))
px.bar(binary_outcome_rates, x='feature', y='outcome_rate_pct', color='category', barmode='group', hover_data=['n'], title='Heart disease/attack prevalence by binary indicator', labels={'outcome_rate_pct':'Outcome prevalence (%)'}).show()

,feature,category,outcome_rate_pct,n
21,AnyHealthcare,0.0,7.01,12417.0
22,AnyHealthcare,1.0,9.54,241263.0
4,CholCheck,0.0,2.86,9470.0
5,CholCheck,1.0,9.67,244210.0
10,Diabetes,0.0,7.18,213703.0
11,Diabetes,1.0,14.34,4631.0
12,Diabetes,2.0,22.29,35346.0
30,DiffWalk,0.0,6.62,211005.0
31,DiffWalk,1.0,23.23,42675.0
34,Education,1.0,16.67,174.0


This view compares outcome prevalence across recorded values for selected ordered survey variables and health-day counts. The x values are original codes: `Age` is an age category, not age in years, and higher `GenHlth` values represent worse self-rated health. `MentHlth` and `PhysHlth` count days in the past 30 days. The BMI plot compares distributions by outcome group; BMI is in kg/m², and extreme values merit checking before modeling.

In [83]:
ordered_cols = ['Diabetes', 'GenHlth', 'Age', 'Education', 'Income', 'MentHlth', 'PhysHlth']
long = brfss_df_selected[ordered_cols + [target]].melt(id_vars=[target], var_name='feature', value_name='value').dropna()
rate_by_value = long.groupby(['feature', 'value'], as_index=False).agg(outcome_rate=(target, 'mean'), n=(target, 'size'))
rate_by_value['outcome_rate_pct'] = rate_by_value['outcome_rate'] * 100
px.line(rate_by_value, x='value', y='outcome_rate_pct', facet_col='feature', facet_col_wrap=3, markers=True, hover_data=['n'], title='Outcome prevalence across coded survey values', labels={'value':'Recorded value / category','outcome_rate_pct':'Outcome prevalence (%)'}).show()
px.histogram(brfss_df_selected, x='BMI', color=target, histnorm='percent', barmode='overlay', opacity=.65, title='BMI distribution by heart disease/attack status').show()

## Correlations and interpretation

Pearson correlations are a quick screening view and can understate or misrepresent relationships for binary and ordinal encoded survey variables.

The sorted table ranks each variable by the absolute size of its Pearson correlation with the binary outcome; the sign indicates direction. The heatmap shows pairwise linear correlation across numeric columns and may help flag related predictors. Since many columns are binary or ordinal codes, these are descriptive screening tools, not a complete measure of association or evidence of causality.

In [84]:
corr = brfss_df_selected.corr(numeric_only=True)
target_corr = corr[target].drop(target).sort_values(key=lambda s: s.abs(), ascending=False)
display(target_corr.to_frame('pearson_r'))
px.imshow(corr, zmin=-1, zmax=1, color_continuous_scale='RdBu_r', aspect='auto', title='Feature correlation matrix').show()

,pearson_r
GenHlth,0.258383
Age,0.221618
DiffWalk,0.212709
HighBP,0.209361
Stroke,0.203002
PhysHlth,0.181698
HighChol,0.180765
Diabetes,0.180272
Income,-0.141011
Smoker,0.114441


## Key EDA insights

- The outcome is uncommon: 25,023 of 295,751 respondents (8.46%) reported heart disease or a heart attack. A model should be assessed with class-sensitive metrics such as recall, precision, PR-AUC, and a confusion matrix rather than accuracy alone.
- Outcome prevalence rises sharply across the age codes, from 0.42% at code 1 to 23.47% at code 13. These are age bands, not single-year ages; this strong pattern also means unadjusted associations for other features may partly reflect age.
- Prevalence increases from 2.00% for best self-rated general health (code 1) to 32.45% for worst (code 5). It is 6.38% for Diabetes code 0, 13.90% for code 1, and 21.78% for code 2; preserve the three-category coding when interpreting diabetes.
- The largest binary-group differences include prior stroke (37.71% vs 7.33%), difficulty walking (22.13% vs 5.87%), high blood pressure (15.85% vs 3.58%), and high cholesterol (15.38% vs 4.79%). These are descriptive, unadjusted comparisons and are not causal effects.
- Respondents reporting the outcome have a higher median BMI (28 vs 27) and mean BMI (29.44 vs 28.11). The distributions overlap, so BMI alone is not a clear separator.
- HighChol is missing for 10.83% of records and BMI for 3.96%; assess missingness handling before modeling. The remaining missing values are negligible in count.
- The binary-indicator chart now limits its calculation to 0/1 fields. `Sex` is coded 1/2, so its mean is not a prevalence; its outcome rates are 10.95% for code 1 and 6.54% for code 2 in this sample. Decode the categories from the data dictionary before assigning labels.
- State-level rates vary, but these raw estimates are unweighted and can reflect differences in respondent composition. Use survey weights/design for population estimates and adjusted analyses for independent associations.



### Reading the results

- Review the missingness table before modeling; missing values change the usable sample size.
- The positive class is much smaller than the negative class, so accuracy alone would be a poor classifier metric.
- Age, general health, diabetes, education, and income are coded categories; do not assume each step represents an equal interval. BMI is a numeric measure; physical and mental health days range from 0 to 30.
- BRFSS sampling weights and design are not included in these simple summaries. Associations are unadjusted and do not imply causation.